In [1]:
import textwrap


def menu():
    menu = """\n
    ================ MENU ================
    [d]\tDepositar
    [s]\tSacar
    [e]\tExtrato
    [nc]\tNova conta
    [lc]\tListar contas
    [nu]\tNovo usuário
    [q]\tSair
    => """
    return input(textwrap.dedent(menu))


def depositar(saldo, valor, extrato, /):
    if valor > 0:
        saldo += valor
        extrato += f"Depósito:\tR$ {valor:.2f}\n"
        print("\n=== Depósito realizado com sucesso! ===")
    else:
        print("\n@@@ Operação falhou! O valor informado é inválido. @@@")

    return saldo, extrato


def sacar(*, saldo, valor, extrato, limite, numero_saques, limite_saques):
    excedeu_saldo = valor > saldo
    excedeu_limite = valor > limite
    excedeu_saques = numero_saques >= limite_saques

    if excedeu_saldo:
        print("\n@@@ Operação falhou! Você não tem saldo suficiente. @@@")

    elif excedeu_limite:
        print("\n@@@ Operação falhou! O valor do saque excede o limite. @@@")

    elif excedeu_saques:
        print("\n@@@ Operação falhou! Número máximo de saques excedido. @@@")

    elif valor > 0:
        saldo -= valor
        extrato += f"Saque:\t\tR$ {valor:.2f}\n"
        numero_saques += 1
        print("\n=== Saque realizado com sucesso! ===")

    else:
        print("\n@@@ Operação falhou! O valor informado é inválido. @@@")

    return saldo, extrato


def exibir_extrato(saldo, /, *, extrato):
    print("\n================ EXTRATO ================")
    print("Não foram realizadas movimentações." if not extrato else extrato)
    print(f"\nSaldo:\t\tR$ {saldo:.2f}")
    print("==========================================")


def criar_usuario(usuarios):
    cpf = input("Informe o CPF (somente número): ")
    usuario = filtrar_usuario(cpf, usuarios)

    if usuario:
        print("\n@@@ Já existe usuário com esse CPF! @@@")
        return

    nome = input("Informe o nome completo: ")
    data_nascimento = input("Informe a data de nascimento (dd-mm-aaaa): ")
    endereco = input("Informe o endereço (logradouro, nro - bairro - cidade/sigla estado): ")

    usuarios.append({"nome": nome, "data_nascimento": data_nascimento, "cpf": cpf, "endereco": endereco})

    print("=== Usuário criado com sucesso! ===")


def filtrar_usuario(cpf, usuarios):
    usuarios_filtrados = [usuario for usuario in usuarios if usuario["cpf"] == cpf]
    return usuarios_filtrados[0] if usuarios_filtrados else None


def criar_conta(agencia, numero_conta, usuarios):
    cpf = input("Informe o CPF do usuário: ")
    usuario = filtrar_usuario(cpf, usuarios)

    if usuario:
        print("\n=== Conta criada com sucesso! ===")
        return {"agencia": agencia, "numero_conta": numero_conta, "usuario": usuario}

    print("\n@@@ Usuário não encontrado, fluxo de criação de conta encerrado! @@@")


def listar_contas(contas):
    for conta in contas:
        linha = f"""\
            Agência:\t{conta['agencia']}
            C/C:\t\t{conta['numero_conta']}
            Titular:\t{conta['usuario']['nome']}
        """
        print("=" * 100)
        print(textwrap.dedent(linha))


def main():
    LIMITE_SAQUES = 3
    AGENCIA = "0001"

    saldo = 0
    limite = 500
    extrato = ""
    numero_saques = 0
    usuarios = []
    contas = []

    while True:
        opcao = menu()

        if opcao == "d":
            valor = float(input("Informe o valor do depósito: "))

            saldo, extrato = depositar(saldo, valor, extrato)

        elif opcao == "s":
            valor = float(input("Informe o valor do saque: "))

            saldo, extrato = sacar(
                saldo=saldo,
                valor=valor,
                extrato=extrato,
                limite=limite,
                numero_saques=numero_saques,
                limite_saques=LIMITE_SAQUES,
            )

        elif opcao == "e":
            exibir_extrato(saldo, extrato=extrato)

        elif opcao == "nu":
            criar_usuario(usuarios)

        elif opcao == "nc":
            numero_conta = len(contas) + 1
            conta = criar_conta(AGENCIA, numero_conta, usuarios)

            if conta:
                contas.append(conta)

        elif opcao == "lc":
            listar_contas(contas)

        elif opcao == "q":
            break

        else:
            print("Operação inválida, por favor selecione novamente a operação desejada.")


main()



================ MENU ================
[d]	Depositar
[s]	Sacar
[e]	Extrato
[nc]	Nova conta
[lc]	Listar contas
[nu]	Novo usuário
[q]	Sair
=> d
Informe o valor do depósito: 10000

=== Depósito realizado com sucesso! ===


================ MENU ================
[d]	Depositar
[s]	Sacar
[e]	Extrato
[nc]	Nova conta
[lc]	Listar contas
[nu]	Novo usuário
[q]	Sair
=> s
Informe o valor do saque: 100

=== Saque realizado com sucesso! ===


================ MENU ================
[d]	Depositar
[s]	Sacar
[e]	Extrato
[nc]	Nova conta
[lc]	Listar contas
[nu]	Novo usuário
[q]	Sair
=> e

================ EXTRATO ================
Depósito:	R$ 10000.00
Saque:		R$ 100.00


Saldo:		R$ 9900.00


================ MENU ================
[d]	Depositar
[s]	Sacar
[e]	Extrato
[nc]	Nova conta
[lc]	Listar contas
[nu]	Novo usuário
[q]	Sair
=> nc
Informe o CPF do usuário: 1234567890

@@@ Usuário não encontrado, fluxo de criação de conta encerrado! @@@


================ MENU ================
[d]	Depositar
[s]	Sacar
[e]

In [2]:
from abc import ABC, abstractclassmethod, abstractproperty
from datetime import datetime

class Cliente:
  def __init__(self, endereco):
    self.endereco = endereco
    self.contas = []

  def realizar_transacao(self, conta, transacao):
    transacao.registrar(conta)

  def adicionar_conta(self, conta):
    self.contas.append(conta)

class PessoaFisica(Cliente):
  def __init__(self, nome, data_nascimento, cpf, endereco):
    super().__init__(endereco)
    self.nome = nome
    self.data_nascimento = data_nascimento
    self.cpf = cpf

class Historico:
  def __init__(self):
    self._transacoes = []

  @property
  def transacoes(self):
    return self._transacoes

  def adicionar_transacao(self, transacao):
    self._transacoes.append({
        "tipo": transacao.__class__.__name__,
        "valor": transacao.valor,
        "data": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    })


class Conta:
  def __init__(self, numero, cliente):
    self._saldo = 0
    self._numero = numero
    self._agencia = "0001"
    self._cliente = cliente
    self._historico = Historico()

  @classmethod
  def nova_conta(cls, cliente, numero):
    return cls(numero, cliente)

  @property
  def saldo(self):
    return self._saldo

  @property
  def numero(self):
    return self._numero

  @property
  def agencia(self):
    return self._agencia

  @property
  def cliente(self):
    return self._cliente

  @property
  def historico(self):
    return self._historico

  def sacar(self, valor):
    saldo = self.saldo
    excedeu_saldo = valor > saldo

    if excedeu_saldo:
      print("\nSaldo insuficiente")

    elif valor > 0:
      self._saldo -= valor
      print("\nSaque realizado com sucesso")
      return True

    else:
      print("\nO valor informado é inválido")

    return False

  def depositar(self, valor):
    if valor > 0:
      self._saldo += valor
      print("Depósito realizado com sucesso")

    else:
      print("O valor informado é inválido")
      return False

    return True

class ContaCorrente(Conta):
  def __init__(self, numero, cliente, limite=500, limite_saques=3):
    super().__init__(numero, cliente)
    self._limite = limite
    self._limite_saques = limite_saques

  def sacar(self, valor):
    numero_saques = len(
        [transacao for transacao in self.historico.
         transacoes if transacao["tipo"] == Saque.__name__]
        )

    excedeu_limite = valor > self._limite
    excedeu_saques = numero_saques >= self._limite_saques

    if excedeu_limite:
      print("O valor do saque excede o limite")

    elif excedeu_saques:
      print("Número máximo de saques excedido")

    else:
      return super().sacar(valor)

    return False

  def __str__(self):
    return f"""\
      Agência:\t{self.agencia}
      C/C:\t\t{self.numero}
      Titular:\t{self.cliente.nome}
    """

class Transacao(ABC):
  @abstractproperty
  def valor(self):
    pass

  @abstractclassmethod
  def registrar(self, conta):
    pass

class Saque(Transacao):
  def __init__(self, valor):
    self._valor = valor

  @property
  def valor(self):
    return self._valor

  def registrar(self, conta):
    sucesso_transacao = conta.sacar(self.valor)

    if sucesso_transacao:
      conta.historico.adicionar_transacao(self)

class Deposito(Transacao):
  def __init__(self, valor):
    self._valor = valor

  @property
  def valor(self):
    return self._valor

  def registrar(self, conta):
    sucesso_transacao = conta.depositar(self.valor)

    if sucesso_transacao:
      conta.historico.adicionar_transacao(self)

In [3]:
from abc import ABC, abstractmethod
from datetime import datetime

class Cliente:
    def __init__(self, endereco):
        self.endereco = endereco
        self.contas = []

    def realizar_transacao(self, conta, transacao):
        transacao.registrar(conta)

    def adicionar_conta(self, conta):
        self.contas.append(conta)

class PessoaFisica(Cliente):
    def __init__(self, nome, data_nascimento, cpf, endereco):
        super().__init__(endereco)
        self.nome = nome
        self.data_nascimento = data_nascimento
        self.cpf = cpf

class Historico:
    def __init__(self):
        self._transacoes = []

    @property
    def transacoes(self):
        return self._transacoes

    def adicionar_transacao(self, transacao):
        self._transacoes.append({
            "tipo": transacao.__class__.__name__,
            "valor": transacao.valor,
            "data": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
        })

class Conta:
    def __init__(self, numero, cliente):
        self._saldo = 0
        self._numero = numero
        self._agencia = "0001"
        self._cliente = cliente
        self._historico = Historico()

    @classmethod
    def nova_conta(cls, cliente, numero):
        return cls(numero, cliente)

    @property
    def saldo(self):
        return self._saldo

    @property
    def numero(self):
        return self._numero

    @property
    def agencia(self):
        return self._agencia

    @property
    def cliente(self):
        return self._cliente

    @property
    def historico(self):
        return self._historico

    def sacar(self, valor):
        saldo = self.saldo
        excedeu_saldo = valor > saldo

        if excedeu_saldo:
            print("\nSaldo insuficiente")
        elif valor > 0:
            self._saldo -= valor
            self._historico.adicionar_transacao(Saque(valor))  # Ensure transaction is logged
            print("\nSaque realizado com sucesso")
            return True
        else:
            print("\nO valor informado é inválido")
        return False

    def depositar(self, valor):
        if valor > 0:
            self._saldo += valor
            self._historico.adicionar_transacao(Deposito(valor))  # Ensure transaction is logged
            print("Depósito realizado com sucesso")
        else:
            print("O valor informado é inválido")
            return False
        return True

class ContaCorrente(Conta):
    def __init__(self, numero, cliente, limite=500, limite_saques=3):
        super().__init__(numero, cliente)
        self._limite = limite
        self._limite_saques = limite_saques

    def sacar(self, valor):
        numero_saques = sum(1 for t in self.historico.transacoes if t["tipo"] == "Saque")

        excedeu_limite = valor > self._limite
        excedeu_saques = numero_saques >= self._limite_saques

        if excedeu_limite:
            print("O valor do saque excede o limite")
        elif excedeu_saques:
            print("Número máximo de saques excedido")
        else:
            return super().sacar(valor)
        return False

    def __str__(self):
        return f"""\nAgência: {self.agencia}
C/C: {self.numero}
Titular: {self.cliente.nome}
"""

class Transacao(ABC):
    @property
    @abstractmethod
    def valor(self):
        pass

    @abstractmethod
    def registrar(self, conta):
        pass

class Saque(Transacao):
    def __init__(self, valor):
        self._valor = valor

    @property
    def valor(self):
        return self._valor

    def registrar(self, conta):
        sucesso_transacao = conta.sacar(self.valor)
        if sucesso_transacao:
            conta.historico.adicionar_transacao(self)

class Deposito(Transacao):
    def __init__(self, valor):
        self._valor = valor

    @property
    def valor(self):
        return self._valor

    def registrar(self, conta):
        sucesso_transacao = conta.depositar(self.valor)
        if sucesso_transacao:
            conta.historico.adicionar_transacao(self)


In [4]:
from abc import ABC, abstractmethod
from datetime import datetime

class Cliente:
    def __init__(self, nome, cpf, endereco):
        self.nome = nome
        self.cpf = cpf
        self.endereco = endereco
        self.contas = []

    def adicionar_conta(self, conta):
        self.contas.append(conta)

    def realizar_transacao(self, conta, transacao):
        transacao.registrar(conta)


class Historico:
    def __init__(self):
        self._transacoes = []

    @property
    def transacoes(self):
        return self._transacoes

    def adicionar_transacao(self, transacao):
        self._transacoes.append({
            "tipo": transacao.__class__.__name__,
            "valor": transacao.valor,
            "data": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
        })


class Conta:
    def __init__(self, numero, cliente):
        self._saldo = 0
        self._numero = numero
        self._agencia = "0001"
        self._cliente = cliente
        self._historico = Historico()

    @classmethod
    def nova_conta(cls, cliente, numero):
        return cls(numero, cliente)

    @property
    def saldo(self):
        return self._saldo

    @property
    def numero(self):
        return self._numero

    @property
    def agencia(self):
        return self._agencia

    @property
    def cliente(self):
        return self._cliente

    @property
    def historico(self):
        return self._historico

    def sacar(self, valor):
        if valor > self._saldo:
            print("\nSaldo insuficiente.")
            return False
        elif valor <= 0:
            print("\nValor inválido para saque.")
            return False
        else:
            self._saldo -= valor
            print("\nSaque realizado com sucesso.")
            return True

    def depositar(self, valor):
        if valor > 0:
            self._saldo += valor
            print("\nDepósito realizado com sucesso.")
            return True
        else:
            print("\nO valor informado é inválido.")
            return False


class ContaCorrente(Conta):
    def __init__(self, numero, cliente, limite=500, limite_saques=3):
        super().__init__(numero, cliente)
        self._limite = limite
        self._limite_saques = limite_saques

    def sacar(self, valor):
        numero_saques = len(
            [t for t in self.historico.transacoes if t["tipo"] == "Saque"]
        )

        if valor > self._limite:
            print("\nValor do saque excede o limite.")
            return False
        elif numero_saques >= self._limite_saques:
            print("\nNúmero máximo de saques excedido.")
            return False
        else:
            return super().sacar(valor)

    def __str__(self):
        return f"Agência: {self.agencia}\nConta: {self.numero}\nTitular: {self.cliente.nome}"


class Transacao(ABC):
    @property
    @abstractmethod
    def valor(self):
        pass

    @abstractmethod
    def registrar(self, conta):
        pass


class Saque(Transacao):
    def __init__(self, valor):
        self._valor = valor

    @property
    def valor(self):
        return self._valor

    def registrar(self, conta):
        sucesso = conta.sacar(self.valor)
        if sucesso:
            conta.historico.adicionar_transacao(self)


class Deposito(Transacao):
    def __init__(self, valor):
        self._valor = valor

    @property
    def valor(self):
        return self._valor

    def registrar(self, conta):
        sucesso = conta.depositar(self.valor)
        if sucesso:
            conta.historico.adicionar_transacao(self)


# ========================
#    INTERFACE DO USUÁRIO
# ========================

clientes = []
contas = []

def criar_cliente():
    nome = input("Nome: ")
    cpf = input("CPF: ")
    endereco = input("Endereço: ")
    cliente = Cliente(nome, cpf, endereco)
    clientes.append(cliente)
    print("\nCliente cadastrado com sucesso!")
    return cliente

def criar_conta(cliente):
    numero_conta = len(contas) + 1
    conta = ContaCorrente(numero_conta, cliente)
    cliente.adicionar_conta(conta)
    contas.append(conta)
    print("\nConta criada com sucesso!")
    return conta

def listar_contas():
    if not contas:
        print("\nNenhuma conta cadastrada.")
    else:
        for conta in contas:
            print(conta)

def buscar_cliente(cpf):
    for cliente in clientes:
        if cliente.cpf == cpf:
            return cliente
    return None

def realizar_deposito():
    cpf = input("Digite o CPF do cliente: ")
    cliente = buscar_cliente(cpf)

    if not cliente:
        print("\nCliente não encontrado.")
        return

    if not cliente.contas:
        print("\nO cliente não possui uma conta.")
        return

    conta = cliente.contas[0]
    valor = float(input("Valor do depósito: "))
    transacao = Deposito(valor)
    cliente.realizar_transacao(conta, transacao)

def realizar_saque():
    cpf = input("Digite o CPF do cliente: ")
    cliente = buscar_cliente(cpf)

    if not cliente:
        print("\nCliente não encontrado.")
        return

    if not cliente.contas:
        print("\nO cliente não possui uma conta.")
        return

    conta = cliente.contas[0]
    valor = float(input("Valor do saque: "))
    transacao = Saque(valor)
    cliente.realizar_transacao(conta, transacao)

def visualizar_extrato():
    cpf = input("Digite o CPF do cliente: ")
    cliente = buscar_cliente(cpf)

    if not cliente:
        print("\nCliente não encontrado.")
        return

    if not cliente.contas:
        print("\nO cliente não possui uma conta.")
        return

    conta = cliente.contas[0]

    print("\n### Extrato ###")
    for transacao in conta.historico.transacoes:
        print(f"{transacao['data']} - {transacao['tipo']}: R$ {transacao['valor']:.2f}")
    print(f"\nSaldo Atual: R$ {conta.saldo:.2f}")

def menu():
    while True:
        print("\n======= MENU =======")
        print("1 - Criar Cliente")
        print("2 - Criar Conta")
        print("3 - Depositar")
        print("4 - Sacar")
        print("5 - Extrato")
        print("6 - Listar Contas")
        print("7 - Sair")

        opcao = input("Escolha uma opção: ")

        if opcao == "1":
            cliente = criar_cliente()
            criar_conta(cliente)
        elif opcao == "2":
            cpf = input("Digite o CPF do cliente: ")
            cliente = buscar_cliente(cpf)
            if cliente:
                criar_conta(cliente)
            else:
                print("\nCliente não encontrado.")
        elif opcao == "3":
            realizar_deposito()
        elif opcao == "4":
            realizar_saque()
        elif opcao == "5":
            visualizar_extrato()
        elif opcao == "6":
            listar_contas()
        elif opcao == "7":
            print("Saindo...")
            break
        else:
            print("Opção inválida.")

menu()



======= MENU =======
1 - Criar Cliente
2 - Criar Conta
3 - Depositar
4 - Sacar
5 - Extrato
6 - Listar Contas
7 - Sair
Escolha uma opção: 1
Nome: Robson da Cruz Augusto
CPF: 123123123123
Endereço: Rua aqui

Cliente cadastrado com sucesso!

Conta criada com sucesso!

======= MENU =======
1 - Criar Cliente
2 - Criar Conta
3 - Depositar
4 - Sacar
5 - Extrato
6 - Listar Contas
7 - Sair
Escolha uma opção: 6
Agência: 0001
Conta: 1
Titular: Robson da Cruz Augusto

======= MENU =======
1 - Criar Cliente
2 - Criar Conta
3 - Depositar
4 - Sacar
5 - Extrato
6 - Listar Contas
7 - Sair
Escolha uma opção: 3
Digite o CPF do cliente: 123123123123
Valor do depósito: 10000

Depósito realizado com sucesso.

======= MENU =======
1 - Criar Cliente
2 - Criar Conta
3 - Depositar
4 - Sacar
5 - Extrato
6 - Listar Contas
7 - Sair
Escolha uma opção: 5
Digite o CPF do cliente: 123123123123

### Extrato ###
24/02/2025 18:36:44 - Deposito: R$ 10000.00

Saldo Atual: R$ 10000.00

======= MENU =======
1 - Criar Client